In [ ]:
"""
Step 2 of the seed-count analysis: compute candidate objectives per simulation run.

Health objectives (computable from agent-level infection-path files, i.e. the
format written by `write_infection_paths`, which is also the format of the
Zenodo file pop8_inf_paths.zip):
    1a) new infections      = Infected(t_max)      - Infected(t_0)       (cumulative)
    1b) new hospitalisations = Hospitalized(t_max) - Hospitalized(t_0)   (cumulative)
    1c) new deaths          = Dead(t_max)          - Dead(t_0)
    2)  years of life lost  = sum over deaths of remaining life expectancy at age of death

Activity objectives (need hourly location logs from your OWN simulations;
they are NOT contained in the Zenodo outputs):
    3) work loss            = 1 - mean_d( W_d / W_baseline(d) )
    4) contact reduction    = 1 - mean_d( rho(C_d) / rho(C_baseline(d)) )
    5) school/event loss    = 1 - mean_{type, d}( X_{type,d} / X_{type,baseline(d)} )

Notes
-----
* All times in the infection-path files are HOURS.
* The ABM draws the whole infection course at the moment of infection, so the
  time-in-state columns can describe states that would only be reached after
  t_max. We therefore always check that a state was entered before t_max.
* Baselines for 3-5 default to the same weekday in the first simulated week.
  Interventions start on day 19, so days 0-6 are intervention-free, and using
  the same weekday avoids counting weekends as "loss".
"""

import numpy as np
import pandas as pd

HOURS_PER_DAY = 24

# mio::abm::LocationType (cpp/models/abm/location_type.h)
HOME, SCHOOL, WORK, SOCIAL_EVENT, BASICS_SHOP, HOSPITAL, ICU = range(7)

# age groups as in determine_age_group() in the pybind code
AGE_BINS = [-1, 4, 15, 34, 59, 79, np.inf]   # 0-4, 5-15, 16-34, 35-59, 60-79, 80+


# --------------------------------------------------------------------------------------
# Loading
# --------------------------------------------------------------------------------------

def load_infection_paths(path):
    """Columns: Agent_id S E I_ns I_sy I_sev I_cri R D (hours)."""
    df = pd.read_csv(path, sep=r"\s+")
    return df.sort_values("Agent_id").reset_index(drop=True)


def load_ages(persons_csv):
    """Exact age per agent. Assumes agent id == row index of the persons file
    (persons are added in file order in initialize_model). Check this once by
    comparing len(ages) with the number of agents in an infection-path file."""
    return pd.read_csv(persons_csv)["age"].to_numpy()


def load_life_table(csv_path):
    """Remaining life expectancy by single year of age, e.g. from the Destatis
    life table (Sterbetafel), averaged over sexes because the ABM has no sex.
    Expected columns: age, remaining_years. Returns an array indexed by age."""
    lt = pd.read_csv(csv_path).sort_values("age")
    max_age = int(lt["age"].max())
    table = np.interp(np.arange(max_age + 1), lt["age"], lt["remaining_years"])
    return table


# --------------------------------------------------------------------------------------
# Objectives 1 and 2: health
# --------------------------------------------------------------------------------------

def health_objectives(paths, t_max_h, ages=None, life_table=None, t0_h=0.0):
    """Objectives 1a-1c and 2 for one simulation run.

    paths      : DataFrame from load_infection_paths
    t_max_h    : end of simulation in hours (e.g. 92 * 24)
    ages       : array of exact ages indexed by agent id (needed for objective 2)
    life_table : array from load_life_table (needed for objective 2)
    """
    eps = 1e-9
    S = paths["S"].to_numpy()

    # Infected: never-infected agents have S == t_max. Initially infected agents have S == 0.
    ever_infected = S < t_max_h - eps
    infected_at_t0 = ever_infected & (S <= t0_h + eps)

    # Hospitalized (InfectedSevere; every critical case passes through severe first).
    t_hosp = S + paths["E"].to_numpy() + paths["I_ns"].to_numpy() + paths["I_sy"].to_numpy()
    reached_severe = (paths["I_sev"].to_numpy() > 0) & ever_infected
    hosp_by_tmax = reached_severe & (t_hosp <= t_max_h + eps)
    hosp_at_t0 = reached_severe & (t_hosp <= t0_h + eps)

    # Dead: D is the time spent dead up to t_max, so D > 0 means died before t_max.
    D = paths["D"].to_numpy()
    dead_by_tmax = D > 0
    t_death = t_max_h - D
    dead_at_t0 = dead_by_tmax & (t_death <= t0_h + eps)

    out = {
        "new_infected": int(ever_infected.sum() - infected_at_t0.sum()),
        "new_hospitalized": int(hosp_by_tmax.sum() - hosp_at_t0.sum()),
        "new_deaths": int(dead_by_tmax.sum() - dead_at_t0.sum()),
    }

    if ages is not None and life_table is not None:
        new_dead = dead_by_tmax & ~dead_at_t0
        ids = paths["Agent_id"].to_numpy()[new_dead]
        age_at_death = np.minimum(ages[ids], len(life_table) - 1).astype(int)
        out["years_of_life_lost"] = float(life_table[age_at_death].sum())

    return out


# --------------------------------------------------------------------------------------
# Helpers for objectives 3-5
# --------------------------------------------------------------------------------------

def _daily_series(occ, loc_type, agg="max"):
    """Daily value for one location type from an hourly occupancy log.

    occ : DataFrame with columns t (hour), loc_type, n_agents
          (total number of agents at all locations of that type in that hour)
    agg : "max"  -> peak number of agents present during the day (people at work/school)
          "sum"  -> agent-hours spent there during the day (better for short events)
    """
    s = occ[occ["loc_type"] == loc_type].copy()
    s["day"] = s["t"] // HOURS_PER_DAY
    daily = s.groupby("day")["n_agents"].agg(agg)
    n_days = int(occ["t"].max() // HOURS_PER_DAY) + 1
    return daily.reindex(range(n_days), fill_value=0).to_numpy(dtype=float)


def _baseline(values, mode="first_week"):
    """Baseline value for every day.
    first_week : same weekday in days 0-6 (pre-intervention)
    first_day  : value of day 0 for every day (literal version of the formula)"""
    if mode == "first_day":
        return np.full_like(values, values[0])
    days = np.arange(len(values))
    return values[days % 7]


def _mean_ratio(values, base, skip_days=0):
    """mean over days of values/base, ignoring days with base == 0 (e.g. schools on weekends)."""
    v, b = values[skip_days:], base[skip_days:]
    mask = b > 0
    return float(np.mean(v[mask] / b[mask]))


# --------------------------------------------------------------------------------------
# Objective 3: work
# --------------------------------------------------------------------------------------

def work_loss(occ, agg="max", baseline="first_week", skip_days=0):
    """1 - mean_d(W_d / W_baseline(d)). 0 = nobody missed work, 1 = nobody worked.
    Note: the formula 1 - sum_d(...) needs the mean (sum / number of days),
    otherwise the value is not between 0 and 1."""
    w = _daily_series(occ, WORK, agg)
    return 1.0 - _mean_ratio(w, _baseline(w, baseline), skip_days)


# --------------------------------------------------------------------------------------
# Objective 5: school and social events
# --------------------------------------------------------------------------------------

def school_event_loss(occ, agg_school="max", agg_event="sum", baseline="first_week", skip_days=0):
    """1 - mean over {School, SocialEvent} and days of X_{type,d} / X_{type,baseline(d)}."""
    ratios = []
    for loc_type, agg in [(SCHOOL, agg_school), (SOCIAL_EVENT, agg_event)]:
        x = _daily_series(occ, loc_type, agg)
        ratios.append(_mean_ratio(x, _baseline(x, baseline), skip_days))
    return 1.0 - float(np.mean(ratios))


# --------------------------------------------------------------------------------------
# Objective 4: contact matrix
# --------------------------------------------------------------------------------------

def daily_contact_matrices(occ_age, pop_per_age, exclude_types=(HOME,)):
    """Age x age co-presence matrix per day.

    occ_age     : DataFrame with columns t, location_id, loc_type, age_group, n
                  (number of agents of an age group at a location in an hour)
    pop_per_age : array with number of agents per age group
    Returns array (n_days, n_age, n_age): C[d, a, b] = hours that an average agent
    of age group a spends together with agents of age group b on day d.
    """
    n_age = len(pop_per_age)
    df = occ_age[~occ_age["loc_type"].isin(exclude_types)]
    X = (df.pivot_table(index=["t", "location_id"], columns="age_group",
                        values="n", aggfunc="sum", fill_value=0)
           .reindex(columns=range(n_age), fill_value=0))
    days = X.index.get_level_values("t").to_numpy() // HOURS_PER_DAY
    n_days = int(occ_age["t"].max() // HOURS_PER_DAY) + 1

    C = np.zeros((n_days, n_age, n_age))
    for d in range(n_days):
        Xd = X.to_numpy()[days == d].astype(float)
        if len(Xd) == 0:
            continue
        M = Xd.T @ Xd - np.diag(Xd.sum(axis=0))        # pairs, without contact with oneself
        C[d] = M / np.asarray(pop_per_age, dtype=float)[:, None]
    return C


def contact_reduction(C, baseline="first_week", skip_days=0):
    """1 - mean_d( rho(C_d) / rho(C_baseline(d)) ), rho = spectral radius.
    The spectral radius is proportional to the reproduction number in a
    next-generation-matrix view, so it measures 'how much transmission
    potential the contact structure allows'."""
    rho = np.array([np.max(np.abs(np.linalg.eigvals(Cd))) for Cd in C])
    return 1.0 - _mean_ratio(rho, _baseline(rho, baseline), skip_days)


def total_contact_reduction(C, baseline="first_week", skip_days=0):
    """Simpler alternative: 1 - mean_d( sum(C_d) / sum(C_baseline(d)) )."""
    tot = C.sum(axis=(1, 2))
    return 1.0 - _mean_ratio(tot, _baseline(tot, baseline), skip_days)


def pop_per_age_group(ages):
    groups = pd.cut(ages, AGE_BINS, labels=False)
    return np.bincount(groups, minlength=len(AGE_BINS) - 1)



# --------------------------------------------------------------------------------------
# Compartment files (pop8_comps.zip, format of write_compartments)
# Header: t S E Ins Isy Isev Icri R D   (t in hours, values = number of agents)
# --------------------------------------------------------------------------------------

def load_compartments(path):
    return pd.read_csv(path, sep=r"\s+").sort_values("t").reset_index(drop=True)


def compartment_objectives(comps):
    """Health objectives that can be computed from compartment sizes alone.

    new_infected      : S(t0) - S(t_max)  -> everyone who left S was newly infected
    new_deaths        : D(t_max) - D(t0)
    peak_hospitalized : max_t (Isev + Icri)  -> hospital beds needed at the peak
    peak_icu          : max_t Icri
    net_change_hosp   : (Isev+Icri)(t_max) - (Isev+Icri)(t0); literal 'end minus start'.
                        Hospitalized is a stock, so this is a net change, can be
                        negative and is NOT a count of hospitalisations. Reported
                        only for completeness. For the cumulative number of
                        hospitalisations use health_objectives() on infection paths.
    """
    hosp = comps["Isev"] + comps["Icri"]
    return {
        "new_infected": int(comps["S"].iloc[0] - comps["S"].iloc[-1]),
        "new_deaths": int(comps["D"].iloc[-1] - comps["D"].iloc[0]),
        "peak_hospitalized": int(hosp.max()),
        "peak_icu": int(comps["Icri"].max()),
        "net_change_hosp": int(hosp.iloc[-1] - hosp.iloc[0]),
    }


# --------------------------------------------------------------------------------------
# Contact files (pop8_contacts.zip, format of write_contact_file)
# Header: loc_type t mean_num_agents max_num_agents
#   t = time step (hours), mean/max over all OCCUPIED locations of that type
# --------------------------------------------------------------------------------------

def load_contacts(path):
    return pd.read_csv(path, sep=r"\s+")


def contact_volume_proxy(contacts, agents_per_type=None, exclude_types=(HOME,),
                         baseline="first_week", skip_days=0):
    """Proxy for objective 4 when no age-structured log is available.

    Per hour and location type, an agent meets on average (mean_num_agents - 1)
    others. Without agent counts the proxy is the sum of (mean - 1) over the
    location types; with agents_per_type (DataFrame t, loc_type, n_agents, e.g.
    from pop8.zip) each type is weighted by the number of agents present.
    Daily value = sum over hours. Returns 1 - mean_d(V_d / V_baseline(d)).

    Caveats: mean is over locations, not over agents (large locations are
    underweighted), and there is no age structure, so no spectral radius.
    """
    c = contacts[~contacts["loc_type"].isin(exclude_types)].copy()
    c["pairs"] = (c["mean_num_agents"] - 1).clip(lower=0)
    if agents_per_type is not None:
        c = c.merge(agents_per_type, on=["t", "loc_type"], how="left").fillna({"n_agents": 0})
        c["pairs"] *= c["n_agents"]
    c["day"] = c["t"] // HOURS_PER_DAY
    n_days = int(c["t"].max() // HOURS_PER_DAY) + 1
    v = c.groupby("day")["pairs"].sum().reindex(range(n_days), fill_value=0).to_numpy(float)
    return 1.0 - _mean_ratio(v, _baseline(v, baseline), skip_days)


# --------------------------------------------------------------------------------------
# pop8.zip location-type files
#   num_agents_loctype.txt     : Sim,t,locType,NumAgents
#   num_agents_loctype_ag.txt  : Sim,t,locType,AgeGroup,NumAgents
# All 100 runs are stacked in one file and identified by the column Sim.
# --------------------------------------------------------------------------------------

LOC_TYPE_NAMES = {"home": HOME, "school": SCHOOL, "work": WORK, "socialevent": SOCIAL_EVENT,
                  "basicsshop": BASICS_SHOP, "hospital": HOSPITAL, "icu": ICU}


def _normalize_loc_type(s):
    """locType may be stored as integer (enum value) or as name ('Work', 'SocialEvent', ...)."""
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(int)
    key = s.astype(str).str.replace(r"[^A-Za-z]", "", regex=True).str.lower()
    out = key.map(LOC_TYPE_NAMES)
    if out.isna().any():
        raise ValueError(f"unknown location types: {sorted(key[out.isna()].unique())}")
    return out.astype(int)


def _detect_t_unit(t, n_days_sim=92):
    """'hours' if there are clearly more time points than simulated days, else 'days'."""
    return "hours" if t.nunique() > 2 * n_days_sim else "days"


def _to_hours(df, t_unit="auto"):
    """Express t in hours so that the helpers above (day = t // 24) work for both
    hourly and daily files. For daily files every day has exactly one value."""
    unit = _detect_t_unit(df["t"]) if t_unit == "auto" else t_unit
    df = df.copy()
    if unit == "days":
        df["t"] = (df["t"] * HOURS_PER_DAY).round().astype(int)
    else:
        df["t"] = df["t"].round().astype(int)
    return df, unit


def load_loctype(path, t_unit="auto"):
    """Returns (DataFrame Sim, t[hours], loc_type, n_agents, detected unit)."""
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    df = df.rename(columns={"locType": "loc_type", "NumAgents": "n_agents"})
    df["loc_type"] = _normalize_loc_type(df["loc_type"])
    return _to_hours(df, t_unit)


def load_loctype_ag(path, t_unit="auto", keep_types=None):
    """Returns (DataFrame Sim, t[hours], loc_type, age_group, n, detected unit).
    keep_types: optionally keep only some location types to save memory (~250 MB file)."""
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    df = df.rename(columns={"locType": "loc_type", "AgeGroup": "age_group", "NumAgents": "n"})
    df["loc_type"] = _normalize_loc_type(df["loc_type"])
    if keep_types is not None:
        df = df[df["loc_type"].isin(keep_types)]
    return _to_hours(df, t_unit)


def activity_objectives(occ_sim, t_unit, baseline="first_week"):
    """Objectives 3 and 5 for one run from num_agents_loctype.
    With hourly data: peak per day for work/school, agent-hours for social events.
    With daily data there is only one value per day, so the aggregation does not matter."""
    agg_work = "max" if t_unit == "hours" else "sum"
    agg_event = "sum"
    return {
        "work_loss": work_loss(occ_sim, agg=agg_work, baseline=baseline),
        "school_event_loss": school_event_loss(occ_sim, agg_school=agg_work,
                                               agg_event=agg_event, baseline=baseline),
    }


# --------------------------------------------------------------------------------------
# Objective 4, approximate age-structured contact matrix
# from num_agents_loctype_ag (who is where) + pop8_contacts (how crowded it is)
# --------------------------------------------------------------------------------------

def approx_contact_matrices(ag_sim, contacts_sim, n_age=6, exclude_types=(HOME,), pop_per_age=None):
    """Proportionate mixing within each location type:

        C_d[a, b] = sum_type  E[a, type, d] * (m[type, d] - 1) * share[b, type, d]  /  N_a

    E[a, type, d]    agent-time of age group a at that type on day d
    m[type, d]       mean number of agents per occupied location (pop8_contacts), daily mean
    share[b, type, d] share of age group b among all agents at that type on day d
    N_a              agents in age group a (by default: all agents at the first time point,
                     because at every time step every agent is at exactly one location)

    Interpretation: hours an average agent of group a spends with agents of group b.
    Limitation: within a location type age groups are assumed to mix in proportion to
    their presence (e.g. pupils are not grouped by class), so assortativity is underestimated.
    """
    ag = ag_sim.copy()
    if pop_per_age is None:
        t0 = ag["t"].min()
        pop_per_age = (ag[ag["t"] == t0].groupby("age_group")["n"].sum()
                       .reindex(range(n_age), fill_value=0).to_numpy(float))
    ag = ag[~ag["loc_type"].isin(exclude_types)]
    ag["day"] = ag["t"] // HOURS_PER_DAY

    k = contacts_sim[~contacts_sim["loc_type"].isin(exclude_types)].copy()
    k["day"] = k["t"] // HOURS_PER_DAY
    m = k.groupby(["day", "loc_type"])["mean_num_agents"].mean().sub(1).clip(lower=0)

    E = ag.groupby(["day", "loc_type", "age_group"])["n"].sum()
    n_days = int(max(ag["day"].max(), k["day"].max())) + 1
    types = sorted(ag["loc_type"].unique())

    C = np.zeros((n_days, n_age, n_age))
    for d in range(n_days):
        for lt in types:
            try:
                e = E.loc[(d, lt)].reindex(range(n_age), fill_value=0).to_numpy(float)
            except KeyError:
                continue
            if e.sum() == 0:
                continue
            mm = m.get((d, lt), 0.0)
            C[d] += np.outer(e, e / e.sum()) * mm
    with np.errstate(divide="ignore", invalid="ignore"):
        C = np.where(pop_per_age[None, :, None] > 0, C / pop_per_age[None, :, None], 0.0)
    return C


# --------------------------------------------------------------------------------------
# Upper bound for the number of seeds (Option 1: one strategy, no CRN)
# --------------------------------------------------------------------------------------

def n_max_table(res, deltas, inflate=1.0):
    """n_max = ceil(8 * (inflate * s)^2 / delta^2) for every objective with a delta.
    inflate > 1 is a safety factor for strategies that are noisier than the 2020 scenario."""
    rows = []
    for col, d in deltas.items():
        s = res[col].std(ddof=1) * inflate
        rows.append({"objective": col, "mean": res[col].mean(), "sd": s, "delta": d,
                     "n_max": int(np.ceil(8 * s ** 2 / d ** 2))})
    return pd.DataFrame(rows).set_index("objective")

# --------------------------------------------------------------------------------------
# Example: health objectives for the 100 Zenodo runs
# --------------------------------------------------------------------------------------

if __name__ == "__main__":
    import glob
    import os
    import re

    def sim_number(path):
        """Run number from a file name like '17_comps.txt'. CHECK that it matches Sim."""
        return int(re.findall(r"\d+", os.path.basename(path))[0])

    comp_files = {sim_number(f): f for f in glob.glob("pop8_comps/*.txt")}
    contact_files = {sim_number(f): f for f in glob.glob("pop8_contacts/*.txt")}

    occ, unit = load_loctype("pop8/num_agents_loctype.txt")
    ag, unit_ag = load_loctype_ag("pop8/num_agents_loctype_ag.txt")
    print("time unit num_agents_loctype:", unit, "| num_agents_loctype_ag:", unit_ag)

    rows = []
    for sim in sorted(occ["Sim"].unique()):
        row = {"Sim": sim}
        row.update(activity_objectives(occ[occ["Sim"] == sim], unit))
        if sim in comp_files:
            row.update(compartment_objectives(load_compartments(comp_files[sim])))
        if sim in contact_files:
            contacts = load_contacts(contact_files[sim])
            row["contact_volume_reduction"] = contact_volume_proxy(contacts)
            C = approx_contact_matrices(ag[ag["Sim"] == sim], contacts)
            row["contact_reduction_rho"] = contact_reduction(C)
        rows.append(row)

    res = pd.DataFrame(rows).set_index("Sim")
    print(res.describe().T)
    res.to_csv("objectives_zenodo.csv")

    # Deltas: smallest difference that matters to you (EXAMPLE values, set your own)
    deltas = {
        "new_infected": 0.005 * 1_536_985,
        "new_deaths": 20,
        "peak_hospitalized": 20,
        "peak_icu": 5,
        "work_loss": 0.01,
        "school_event_loss": 0.01,
        "contact_reduction_rho": 0.01,
    }
    deltas = {k: v for k, v in deltas.items() if k in res.columns}
    print(n_max_table(res, deltas, inflate=1.5))

    # Optional, objective 2 (years of life lost) from pop8_inf_paths + persons.csv:
    # ages = load_ages("persons.csv")
    # life_table = load_life_table("life_table.csv")   # TODO: fill from Destatis
    # for f in sorted(glob.glob("pop8_inf_paths/*.txt")):
    #     h = health_objectives(load_infection_paths(f), 92 * HOURS_PER_DAY, ages, life_table)